# Data pipeline — orchestrator

1. **Step 01 — Prediction contract:** load `config/data_contract.yaml` and test it against the source data. The pipeline should not run on a contract that fails.
2. **Pipeline stages:** run the layer notebooks in `util/` in order (Bronze → Silver → Gold).

The contract is the single source of truth: the layer notebooks read their sources, keys, PII list and (later) label definition from it instead of defining them locally.

In [1]:
import os, sys, csv, math
from datetime import date

import yaml
import pandas as pd

# Project folder: /app inside the Docker container, otherwise the parent of this notebook's folder (script/)
PROJECT_DIR   = os.environ.get("PROJECT_DIR") or ("/app" if os.path.isdir("/app/data") else os.path.abspath(".."))
RAW_DIR       = os.path.join(PROJECT_DIR, "data")
CONTRACT_PATH = os.path.join(PROJECT_DIR, "config", "data_contract.yaml")


def load_contract(path=CONTRACT_PATH):
    with open(path, encoding="utf-8") as f:
        return yaml.safe_load(f)


CONTRACT = load_contract()

## Step 01 — Prediction contract

In [2]:
c, t = CONTRACT, CONTRACT["target"]
pd.DataFrame([
    ("Contract",           f"{c['contract']['name']} v{c['contract']['version']} ({c['contract']['status']})"),
    ("Problem",            " ".join(c["problem"]["description"].split())),
    ("Entity key",         c["entity"]["key"]),
    ("Prediction grain",   c["prediction_grain"]),
    ("Scoring timestamp",  f"{c['scoring_timestamp']['table']}.{c['scoring_timestamp']['column']}"),
    ("Target",             f"{t['name']} = {t['label_def']}  (1 if DPD >= {t['dpd_threshold']} at MOB {t['mob']})"),
    ("Positive class",     t["positive_class"]),
    ("Outcome horizon",    f"{c['outcome_horizon_months']} months"),
    ("Eligibility",        "; ".join(e["rule"] for e in c["eligibility"])),
    ("Feature tables",     ", ".join(k for k, v in c["feature_availability"]["tables"].items() if v["allowed"])),
    ("Not allowed",        ", ".join(f"{k} ({v['reason'].split(';')[0]})"
                                     for k, v in c["feature_availability"]["tables"].items() if not v["allowed"])),
    ("Excluded (PII)",     ", ".join(f"{tb}.{col}" for tb, cols in c["excluded_columns"]["pii"].items() for col in cols)),
], columns=["Field", "Value"]).style.hide(axis="index")

Field,Value
Contract,loan_default_30dpd_6mob v1.1.0 (draft)
Problem,"At loan application, predict whether the loan will be 30 or more days past due at month-on-book 6 (six months after the loan starts)."
Entity key,loan_id
Prediction grain,"one row per loan_id, scored once at application"
Scoring timestamp,lms_loan_daily.loan_start_date
Target,label = 30dpd_6mob (1 if DPD >= 30 at MOB 6)
Positive class,1
Outcome horizon,6 months
Eligibility,"exactly one row per loan_id in the model table; loan has a snapshot with installment_num = target.mob (otherwise label unknown -> excluded); loans whose MOB-6 date is after the data cut-off are excluded, never labelled 0"
Feature tables,"features_attributes, features_financials, feature_clickstream"


### Contract helpers

- `source_header(table)` — column names of a source file.
- `allowed_predictors()` — every column that may be used as a feature under the contract.
- `check_predictors(columns)` — raises an error naming every column that is **not available at scoring time** (label-source columns, PII, identifiers, unknown columns). Gold and the ML pipeline call this on their feature lists.
- `as_of_join(...)` — reference point-in-time join: for each loan, the latest feature snapshot dated on or before the scoring timestamp.

In [3]:
def source_header(table):
    with open(os.path.join(RAW_DIR, CONTRACT["sources"][table]["file"]), newline="", encoding="utf-8") as f:
        return next(csv.reader(f))


def excluded_reasons(contract=CONTRACT):
    # column -> reason it can never be a predictor
    reasons = {col: "identifier" for col in contract["excluded_columns"]["identifiers"]}
    for table, cols in contract["excluded_columns"]["pii"].items():
        reasons.update({col: f"PII ({table})" for col in cols})
    for table, cols in contract["excluded_columns"].get("replaced_by_derived", {}).items():
        reasons.update({col: f"raw text replaced by derived feature ({table})" for col in cols})
    return reasons


def allowed_predictors(contract=CONTRACT):
    excluded = excluded_reasons(contract)
    out = {}
    for table, rule in contract["feature_availability"]["tables"].items():
        if rule["allowed"]:
            src = contract["sources"][table]
            for col in source_header(table):
                if col not in excluded and col not in src["keys"] and col != src["event_time"]:
                    out[col] = table
            for col in contract.get("derived_features", {}).get(table, {}):
                out[col] = table
    return out          # column -> source table


def check_predictors(columns, contract=CONTRACT):
    allowed = allowed_predictors(contract)
    excluded = excluded_reasons(contract)
    blocked = {t: r["reason"] for t, r in contract["feature_availability"]["tables"].items() if not r["allowed"]}
    blocked_cols = {col: t for t in blocked for col in source_header(t)}
    rejected = {}
    for col in columns:
        if col in allowed:
            continue
        if col in excluded:
            rejected[col] = excluded[col]
        elif col in blocked_cols:
            rejected[col] = f"from {blocked_cols[col]}: not available at scoring time"
        else:
            rejected[col] = "not defined in any allowed feature table"
    if rejected:
        raise ValueError("Predictors rejected by the data contract: " +
                         "; ".join(f"{k} -> {v}" for k, v in rejected.items()))
    return True


def as_of_join(loans, features, key, scoring_col, event_col):
    # Latest feature row with event_col <= scoring_col, per loan. Rows dated after scoring time are never used.
    m = loans.merge(features, on=key, how="left", suffixes=("", "_feat"))
    m = m[m[event_col].isna() | (m[event_col] <= m[scoring_col])]
    m = m.sort_values(event_col).groupby(list(loans.columns), dropna=False, as_index=False).tail(1)
    return loans.merge(m, on=list(loans.columns), how="left")

### Contract tests

| Group | Tests |
|---|---|
| A. Structure | required fields present; target internally consistent (classes, threshold, horizon = MOB, `label_def`); sign-off status |
| B. Sources | files exist; keys, event-time, scoring and target-derivation columns exist; feature rules reference real tables and join keys; label source is not a feature table; PII columns exist |
| C. Predictor rule | allowed predictors pass; label-source columns, PII and identifiers are rejected (checklist: *reject predictors unavailable at scoring time*) |
| D. Data | one row per loan; MOB = months since loan start; every loan without a MOB-6 snapshot is genuinely immature; point-in-time join never picks a future-dated row (synthetic future-event test); feature coverage at scoring time |

In [4]:
results = []

def run_test(group, name, fn):
    try:
        out = fn()
        status, detail = out if isinstance(out, tuple) else ("PASS", out or "")
    except Exception as e:
        status, detail = "FAIL", f"{type(e).__name__}: {e}"
    results.append(dict(group=group, test=name, status=status, detail=detail))


# ---------- A. Structure ----------
REQUIRED = ["contract.name", "contract.version", "contract.status", "problem.type", "entity.key",
            "prediction_grain", "scoring_timestamp.table", "scoring_timestamp.column",
            "target.name", "target.label_def", "target.positive_class", "target.negative_class",
            "target.dpd_threshold", "target.mob", "target.table", "outcome_horizon_months",
            "eligibility", "sources", "feature_availability.tables", "excluded_columns.pii",
            "excluded_columns.identifiers"]

def t_required():
    missing = []
    for path in REQUIRED:
        node = CONTRACT
        for part in path.split("."):
            if not isinstance(node, dict) or part not in node or node[part] in (None, "", []):
                missing.append(path)
                break
            node = node[part]
    assert not missing, f"missing fields: {missing}"
    return f"{len(REQUIRED)} required fields present"

def t_target():
    tg = CONTRACT["target"]
    assert {tg["positive_class"], tg["negative_class"]} == {0, 1}, "classes must be 0 and 1"
    assert isinstance(tg["dpd_threshold"], int) and tg["dpd_threshold"] > 0, "dpd_threshold must be a positive int"
    assert isinstance(tg["mob"], int) and tg["mob"] > 0, "mob must be a positive int"
    assert CONTRACT["outcome_horizon_months"] == tg["mob"], "outcome_horizon_months must equal target.mob"
    expected = f"{tg['dpd_threshold']}dpd_{tg['mob']}mob"
    assert tg["label_def"] == expected, f"label_def {tg['label_def']} != {expected}"
    return f"{tg['label_def']}, positive class = {tg['positive_class']}"

def t_signoff():
    k = CONTRACT["contract"]
    if k["status"] == "signed":
        assert k["signed_by"] and k["signed_date"], "signed contract needs signed_by and signed_date"
        return f"signed by {k['signed_by']} on {k['signed_date']}"
    return "WARN", f"status = {k['status']}: fill signed_by / signed_date and set status: signed once approved"

run_test("A. Structure", "required_fields", t_required)
run_test("A. Structure", "target_consistent", t_target)
run_test("A. Structure", "sign_off", t_signoff)


# ---------- B. Sources ----------
def t_source_files():
    out = []
    for table, s in CONTRACT["sources"].items():
        path = os.path.join(RAW_DIR, s["file"])
        assert os.path.exists(path), f"{table}: {path} not found"
        hdr = source_header(table)
        missing = [c for c in s["keys"] + [s["event_time"]] if c not in hdr]
        assert not missing, f"{table}: key/event-time columns not in file: {missing}"
        out.append(table)
    return f"{len(out)} sources: files exist, keys and event_time present"

def t_scoring_and_target_columns():
    st, tg = CONTRACT["scoring_timestamp"], CONTRACT["target"]
    assert st["column"] in source_header(st["table"]), f"scoring column {st['column']} not in {st['table']}"
    needed = ["installment_num", "overdue_amt", "due_amt", "snapshot_date", CONTRACT["entity"]["key"]]
    missing = [c for c in needed if c not in source_header(tg["table"])]
    assert not missing, f"target derivation columns missing in {tg['table']}: {missing}"
    return f"{st['table']}.{st['column']} and derivation columns present"

def t_feature_rules():
    fa = CONTRACT["feature_availability"]["tables"]
    for table, rule in fa.items():
        assert table in CONTRACT["sources"], f"{table} is not a declared source"
        if rule["allowed"]:
            assert rule.get("as_of"), f"{table}: allowed table needs an as_of rule"
            assert rule["join_key"] in source_header(table), f"{table}: join key {rule['join_key']} missing"
            assert CONTRACT["sources"][table]["role"] == "feature_source", f"{table}: role must be feature_source"
    label_src = CONTRACT["target"]["table"]
    assert not fa[label_src]["allowed"], f"label source {label_src} must not be a feature table"
    return f"{sum(r['allowed'] for r in fa.values())} feature tables with as-of rules; label source blocked"

def t_pii_columns_exist():
    for table, cols in CONTRACT["excluded_columns"]["pii"].items():
        missing = [col for col in cols if col not in source_header(table)]
        assert not missing, f"PII columns not found in {table}: {missing} (contract out of date?)"
    return "PII columns found in their tables"

run_test("B. Sources", "source_files_and_keys", t_source_files)
run_test("B. Sources", "scoring_and_target_columns", t_scoring_and_target_columns)
run_test("B. Sources", "feature_availability_rules", t_feature_rules)
def t_derived_features():
    fa = CONTRACT["feature_availability"]["tables"]
    n = 0
    for table, feats in CONTRACT.get("derived_features", {}).items():
        assert fa.get(table, {}).get("allowed"), f"derived features declared on non-feature table {table}"
        hdr = source_header(table)
        for name, spec in feats.items():
            missing = [c for c in spec["from"] if c not in hdr]
            assert not missing, f"{table}.{name}: source columns not found: {missing}"
            assert name not in hdr, f"{table}.{name}: derived name clashes with a source column"
            n += 1
    for table, cols in CONTRACT["excluded_columns"].get("replaced_by_derived", {}).items():
        sources = {c for spec in CONTRACT["derived_features"].get(table, {}).values() for c in spec["from"]}
        assert set(cols) <= sources, f"{table}: replaced columns {set(cols) - sources} have no derived replacement"
    return f"{n} derived features, all built from allowed source columns; replaced raw text has a derived version"

run_test("B. Sources", "pii_columns_exist", t_pii_columns_exist)
run_test("B. Sources", "derived_features_valid", t_derived_features)


# ---------- C. Predictor rule ----------
def t_allowed_pass():
    allowed = allowed_predictors()
    check_predictors(list(allowed))
    label_cols = set(source_header(CONTRACT["target"]["table"]))
    clash = sorted(set(allowed) & label_cols)
    assert not clash, f"allowed predictors share names with label-source columns: {clash}"
    by_table = pd.Series(allowed).value_counts().to_dict()
    return f"{len(allowed)} allowed predictors {by_table}"

def t_reject(columns, must_reject):
    def fn():
        try:
            check_predictors(columns)
        except ValueError as e:
            missed = [c for c in must_reject if f"{c} ->" not in str(e)]
            assert not missed, f"not rejected: {missed}"
            return f"rejected {must_reject}"
        raise AssertionError(f"{columns} was accepted")
    return fn

run_test("C. Predictor rule", "allowed_predictors_pass", t_allowed_pass)
run_test("C. Predictor rule", "reject_label_source_columns",
         t_reject(["fe_1", "overdue_amt", "balance", "paid_amt"], ["overdue_amt", "balance", "paid_amt"]))
run_test("C. Predictor rule", "reject_pii", t_reject(["Age", "SSN", "Name"], ["SSN", "Name"]))
run_test("C. Predictor rule", "reject_replaced_raw_text",
         t_reject(["credit_history_age_months", "Type_of_Loan", "Credit_History_Age"], ["Type_of_Loan", "Credit_History_Age"]))
run_test("C. Predictor rule", "reject_identifiers_and_unknown",
         t_reject(["Customer_ID", "loan_start_date", "not_a_column"], ["Customer_ID", "loan_start_date", "not_a_column"]))

In [5]:
# ---------- D. Data (raw source files, independent of the pipeline) ----------
tg, key = CONTRACT["target"], CONTRACT["entity"]["key"]
lms_file = os.path.join(RAW_DIR, CONTRACT["sources"][tg["table"]]["file"])
lms = pd.read_csv(lms_file, dtype=str, usecols=[key, "Customer_ID", "loan_start_date", "installment_num", "snapshot_date"])
lms["installment_num"] = lms["installment_num"].astype(int)
loans = lms.drop_duplicates(key)[[key, "Customer_ID", "loan_start_date"]]
cutoff = lms["snapshot_date"].max()

def t_one_row_per_loan():
    assert loans[key].is_unique, "loan_id not unique"
    per_cust = loans.groupby("Customer_ID")[key].nunique()
    starts = lms.groupby(key)["loan_start_date"].nunique()
    assert (starts == 1).all(), f"{(starts > 1).sum()} loans with more than one loan_start_date"
    assert per_cust.max() == 1, f"{(per_cust > 1).sum()} customers with several loans (customer_key note is wrong)"
    return f"{len(loans):,} loans, one loan_start_date each, one loan per customer"

def t_mob_definition():
    s, d = pd.to_datetime(lms["loan_start_date"]), pd.to_datetime(lms["snapshot_date"])
    months = (d.dt.year - s.dt.year) * 12 + (d.dt.month - s.dt.month)
    bad = (months != lms["installment_num"]).sum()
    assert bad == 0, f"{bad} rows where installment_num != months since loan start"
    return "installment_num = months since loan_start_date in every row"

def t_label_maturity():
    has_mob = set(lms.loc[lms["installment_num"] == tg["mob"], key])
    mob_date = pd.to_datetime(loans["loan_start_date"]) + pd.DateOffset(months=tg["mob"])
    immature = set(loans.loc[mob_date > pd.Timestamp(cutoff), key])
    unexplained = set(loans[key]) - has_mob - immature
    assert not unexplained, f"{len(unexplained)} mature loans have no MOB-{tg['mob']} snapshot"
    return f"{len(has_mob):,} loans with mature label, {len(immature):,} immature (excluded); data cut-off {cutoff}"

def t_future_event_excluded():
    # Synthetic check: a feature row dated after the scoring timestamp must never be selected
    L = pd.DataFrame({"loan_id": ["L1", "L2"], "Customer_ID": ["C1", "C2"],
                      "loan_start_date": ["2024-03-01", "2024-03-01"]})
    Fx = pd.DataFrame({"Customer_ID": ["C1", "C1", "C1", "C2"],
                       "snapshot_date": ["2024-01-01", "2024-03-01", "2024-04-01", "2024-05-01"],
                       "x": ["past", "same_day", "FUTURE", "FUTURE"]})
    j = as_of_join(L, Fx, "Customer_ID", "loan_start_date", "snapshot_date").set_index("loan_id")
    assert j.loc["L1", "x"] == "same_day", f"L1 picked {j.loc['L1', 'x']!r}, expected the latest non-future row"
    assert pd.isna(j.loc["L2", "x"]), f"L2 picked {j.loc['L2', 'x']!r}; only future rows exist, expected none"
    assert len(j) == len(L), "join changed the number of loans"
    return "as-of join picks the latest row <= scoring time and never a future-dated row"

def t_feature_coverage():
    rows = []
    for table, rule in CONTRACT["feature_availability"]["tables"].items():
        if not rule["allowed"]:
            continue
        src = CONTRACT["sources"][table]
        f = pd.read_csv(os.path.join(RAW_DIR, src["file"]), dtype=str, usecols=[rule["join_key"], src["event_time"]])
        j = as_of_join(loans, f, rule["join_key"], "loan_start_date", src["event_time"])
        assert len(j) == len(loans), f"{table}: as-of join changed the number of loans"
        future_rows = f.merge(loans, on=rule["join_key"])
        future_rows = (future_rows[src["event_time"]] > future_rows["loan_start_date"]).sum()
        rows.append(f"{table}: {j[src['event_time']].notna().mean() * 100:.1f}% of loans covered, "
                    f"{future_rows:,} future-dated rows excluded")
    return " | ".join(rows)

run_test("D. Data", "one_row_per_loan", t_one_row_per_loan)
run_test("D. Data", "mob_equals_months_since_start", t_mob_definition)
run_test("D. Data", "label_maturity", t_label_maturity)
run_test("D. Data", "future_event_excluded", t_future_event_excluded)
run_test("D. Data", "feature_coverage_at_scoring_time", t_feature_coverage)

In [6]:
STATUS_STYLE = {"PASS": ("✔ PASS", "#e3f4e3"), "WARN": ("▲ WARN", "#fdf0d2"), "FAIL": ("✖ FAIL", "#f8dede")}
report = pd.DataFrame(results)
view = report.assign(status=report["status"].map(lambda s: STATUS_STYLE[s][0]))
display(view.style.apply(lambda col: [f"background-color: {STATUS_STYLE[v.split()[-1]][1]}" for v in col],
                         subset=["status"]).hide(axis="index"))

failed = report[report["status"] == "FAIL"]
assert failed.empty, f"Contract tests failed: {failed['test'].tolist()}"
print(f"Contract OK: {(report['status'] == 'PASS').sum()} passed, {(report['status'] == 'WARN').sum()} warning(s)")

group,test,status,detail
A. Structure,required_fields,✔ PASS,21 required fields present
A. Structure,target_consistent,✔ PASS,"30dpd_6mob, positive class = 1"
A. Structure,sign_off,▲ WARN,status = draft: fill signed_by / signed_date and set status: signed once approved
B. Sources,source_files_and_keys,✔ PASS,"4 sources: files exist, keys and event_time present"
B. Sources,scoring_and_target_columns,✔ PASS,lms_loan_daily.loan_start_date and derivation columns present
B. Sources,feature_availability_rules,✔ PASS,3 feature tables with as-of rules; label source blocked
B. Sources,pii_columns_exist,✔ PASS,PII columns found in their tables
B. Sources,derived_features_valid,✔ PASS,"28 derived features, all built from allowed source columns; replaced raw text has a derived version"
C. Predictor rule,allowed_predictors_pass,✔ PASS,"68 allowed predictors {'features_financials': 44, 'feature_clickstream': 20, 'features_attributes': 4}"
C. Predictor rule,reject_label_source_columns,✔ PASS,"rejected ['overdue_amt', 'balance', 'paid_amt']"


Contract OK: 17 passed, 1 warning(s)


## Pipeline stages

Each layer is developed in `notebooks/dataprocessing_<layer>.ipynb` and converted to the callable module `util/dataprocessing_<layer>.py`, which the pipeline imports. Set a stage to `False` to skip it; stages are added as the steps are completed.

In [7]:
sys.path.insert(0, PROJECT_DIR)                     # so `util` can be imported
from util import dataprocessing_bronze as bronze
from util import dataprocessing_silver as silver
from util import dataprocessing_gold as gold

RUN_STAGES = {
    "bronze": True,     # Steps 02-04: ingest (skipped if sources unchanged) + audit + raw EDA     (~40 s)
    "silver": True,     # Steps 05-09: schema, validation, cleaning, transforms, Bronze-vs-Silver EDA (~2 min)
    "gold":   True,     # Steps 10-13: feature store + label store, released only if acceptance passes (~1.5 min)
}
BRONZE_OPTIONS = dict(force=False, eda=True, charts=True, html=False, tests=False)
SILVER_OPTIONS = dict(eda=True, charts=True, tests=False)
GOLD_OPTIONS = dict(charts=True, tests=False)
# Gold EDA (steps 14, 16, 15) needs the step-17 split and runs in script/mlpipeline.ipynb

spark = bronze.get_spark("datapipeline")

### Bronze (steps 02–04)

In [8]:
if RUN_STAGES["bronze"]:
    out_bronze = bronze.run(spark, **BRONZE_OPTIONS)
    display(out_bronze["ingestion"][["table", "status", "source_rows", "bronze_rows", "event_partitions",
                                     "min_event_date", "max_event_date"]].style.hide(axis="index"))
    print("Audit:", out_bronze["audit_report"]["status"].value_counts().to_dict())
    print("Raw EDA issues:", out_bronze["issues"]["severity"].value_counts().to_dict())
else:
    print("skipped bronze")

table,status,source_rows,bronze_rows,event_partitions,min_event_date,max_event_date
lms_loan_daily,SKIPPED_UNCHANGED,137500,137500,35,2023-01-01,2025-11-01
feature_clickstream,SKIPPED_UNCHANGED,215376,215376,24,2023-01-01,2024-12-01
features_attributes,SKIPPED_UNCHANGED,12500,12500,25,2023-01-01,2025-01-01
features_financials,SKIPPED_UNCHANGED,12500,12500,25,2023-01-01,2025-01-01


Audit: {'PASS': 33, 'WARN': 3}
Raw EDA issues: {'medium': 21, 'high': 11, 'low': 3}


### Silver (steps 05–09)

In [9]:
if RUN_STAGES["silver"]:
    out_silver = silver.run(spark, **SILVER_OPTIONS)
    v = out_silver["validation_report"]
    display(out_silver["row_counts"].style.hide(axis="index"))
    print("Validation before cleaning:", v[v["stage"] == "pre_clean"]["status"].value_counts().to_dict())
    print("Validation after cleaning: ", v[v["stage"] == "post_clean"]["status"].value_counts().to_dict())
    print("Decisions:", out_silver["decisions"]["decision"].value_counts().to_dict())
    if "step08" in out_silver:
        print("Step 08 (Silver vs Bronze):", out_silver["step08"]["comparison"]["status"].value_counts().to_dict())
else:
    print("skipped silver")

table,bronze_rows,quarantined,rows_into_cleaning,duplicates_removed,silver_rows
lms_loan_daily,137500,0,137500,0,137500
feature_clickstream,215376,0,215376,0,215376
features_attributes,12500,0,12500,0,12500
features_financials,12500,0,12500,0,12500


Validation before cleaning: {'PASS': 83, 'FAIL': 14, 'WARN': 6}
Validation after cleaning:  {'PASS': 97, 'WARN': 6}
Decisions: {'transform': 20, 'defer': 7, 'correct': 5, 'accept': 4, 'exclude': 3}
Step 08 (Silver vs Bronze): {'PASS': 45, 'EXPLAINED': 14}


### Gold (steps 10–13)

In [10]:
if RUN_STAGES["gold"]:
    out_gold = gold.run_build(spark, **GOLD_OPTIONS)
    display(out_gold["join_diagnostics"][["table", "matched_pct", "unmatched_pct", "future_rows_excluded"]]
            .style.hide(axis="index").format({"matched_pct": "{:.1f}%", "unmatched_pct": "{:.1f}%"}))
    display(out_gold["label_summary"].style.hide(axis="index"))
    print("Gold acceptance:", out_gold["acceptance"]["status"].value_counts().to_dict(),
          f"| released feature_store/{gold.FEATURE_VERSION} and label_store/{gold.LABEL_DEF}")
else:
    print("skipped gold")

table,matched_pct,unmatched_pct,future_rows_excluded
features_attributes,100.0%,0.0%,0
features_financials,100.0%,0.0%,0
feature_clickstream,71.8%,28.2%,130468


label_def,maturity_status,loans,label_1,label_rate_pct,data_cutoff
30dpd_6mob,mature,12500,3602,28.816000,2025-11-01


Gold acceptance: {'PASS': 15} | released feature_store/v1 and label_store/30dpd_6mob
